# CaptionGene – Week 2 full caption run on a Colab GPU

Runs the target models (**BLIP-2** and **Qwen2.5-VL**) on all 2,410 images, computes CLIPScore and writes
`captions_out.zip`, which you copy back to the Mac.

**Before you start**
1. On the Mac: `bash scripts/make_colab_bundle.sh` → `~/CaptionGene_colab.zip`
2. Upload that zip to Google Drive into a folder **`MyDrive/CaptionGene/`**
3. Here: *Runtime → Change runtime type → **T4 GPU*** (or L4 / A100 if you have them), then run the cells top to bottom.

**If Colab disconnects:** captions are saved to Google Drive after every batch. Reconnect, run cells 1–4 again and
then the full run (cell 6). Finished captions are skipped, so the run continues where it stopped.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive and choose settings

* `QWEN`: `qwen2.5-vl-3b` fits a 16 GB T4. On an A100 (40 GB+) you can use `qwen2.5-vl-7b`, the model named in the brief.
* `SUBSET`: `all` (2,410 images, ~3–5 h on a T4) or `split:test` (343 images, ~30–45 min) for a quicker first run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/CaptionGene'   # where you uploaded the zip
QWEN = 'qwen2.5-vl-3b'
SUBSET = 'all'
BLIP2_BS, QWEN_BS = 16, 8                           # lower these if you get "CUDA out of memory"

import os
os.environ.update(QWEN=QWEN, SUBSET=SUBSET, BLIP2_BS=str(BLIP2_BS), QWEN_BS=str(QWEN_BS), DRIVE_DIR=DRIVE_DIR)
assert os.path.exists(f'{DRIVE_DIR}/CaptionGene_colab.zip'), 'upload CaptionGene_colab.zip to MyDrive/CaptionGene/ first'

## 3. Unpack the project (captions are written straight to Google Drive)

In [ ]:
%%bash
set -e
rm -rf /content/CaptionGene && mkdir -p /content/CaptionGene
unzip -q "$DRIVE_DIR/CaptionGene_colab.zip" -d /content/CaptionGene
mkdir -p "$DRIVE_DIR/captions_out"
rm -rf /content/CaptionGene/data/captions
ln -s "$DRIVE_DIR/captions_out" /content/CaptionGene/data/captions
echo "images: $(find /content/CaptionGene/data/images -name '*.jpg' | wc -l)"
echo "captions already on Drive:"; ls -l "$DRIVE_DIR/captions_out"

## 4. Install libraries (Colab already has PyTorch)

In [ ]:
!pip -q install "transformers==5.21.0" "accelerate==1.15.0" pyyaml jsonschema tqdm num2words
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

## 5. Smoke test (2 images per model, ~5 min including the model downloads)

Check that the printed captions look like real sentences before starting the long run.

In [ ]:
%%bash
cd /content/CaptionGene/scripts
python 08_generate_captions.py --models blip2 --subset "$SUBSET" --limit 2
python 08_generate_captions.py --models "$QWEN" --subset "$SUBSET" --limit 2
python - <<'PY'
import json, os
for m in ("blip2", os.environ["QWEN"]):
    print(f"--- {m}")
    for line in list(open(f"../data/captions/{m}.jsonl"))[:6]:
        r = json.loads(line)
        print(f"{r.get('object_id') or r['image_id']}: {r['text']}")
PY

## 6. Full run

Template baseline → BLIP-2 → Qwen2.5-VL → CLIPScore → merge → report → schema check.
Keep the tab open. If it disconnects, rerun cells 1–4 and this cell; it resumes.

In [ ]:
!cd /content/CaptionGene/scripts && bash run_week2_gpu.sh

## 7. Results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('/content/CaptionGene/data/reports/captions_week2.md').read()[:6000]))
display(Image('/content/CaptionGene/data/reports/caption_examples.jpg'))

## 8. Package the results for the Mac

Writes `captions_out.zip` to `MyDrive/CaptionGene/` and also downloads it. On the Mac:

```bash
bash scripts/import_colab_results.sh ~/Downloads/captions_out.zip
```

In [ ]:
%%bash
cd /content/CaptionGene/data
rm -f /content/captions_out.zip
zip -q -j /content/captions_out.zip captions/*.jsonl
cp /content/captions_out.zip "$DRIVE_DIR/"
ls -lh /content/captions_out.zip

In [ ]:
from google.colab import files
files.download('/content/captions_out.zip')